In [7]:
# hyperparameters
batch_size = 8
block_size = 128
max_iters = 500
eval_interval = 50
learning_rate = 1e-4
device = 'cpu'
eval_iters = 20
n_embd = 512
n_head = 8
n_layer = 6
dropout = 0.2


In [8]:
import torch
from LoRA import LoraGPT
from tqdm import tqdm

# load checkpoint
checkpoint = torch.load('gpt_checkpoint.pt', map_location='cpu', weights_only=False)

r = 8
lora_alpha = 16

lora_model = LoraGPT(r,lora_alpha)
lora_model.load_state_dict(checkpoint['model_state_dict'], strict=False)

for param in lora_model.parameters():
    param.requires_grad = False

In [9]:
for name, param in lora_model.named_parameters():
    if name.endswith('.A') or name.endswith('.B'):
        param.requires_grad = True

In [10]:
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f"Trainable: {trainable:,}")
print(f"Total: {total:,}")
print(f"Trainable %: {100 * trainable / total:.2f}%")

Trainable: 663,552
Total: 19,776,577
Trainable %: 3.36%


In [11]:
torch.manual_seed(1337)
with open('finetune.txt', 'r') as f:
    text = f.read()

# all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)

# create mappings from characters to integers and vice versa
stoi = checkpoint['stoi'] #string to integer
itos = checkpoint['itos'] #integer to string

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder:

text = ''.join([c for c in text if c in stoi])

# train test splits

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    lora_model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = lora_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    lora_model.train()
    return out


In [12]:
m = lora_model.to(device)

optimizer = torch.optim.AdamW(lora_model.parameters(), lr=learning_rate)

for iter in tqdm(range(max_iters), desc="Training"):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        tqdm.write(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


Training:   0%|          | 0/500 [00:21<?, ?it/s]

step 0: train loss 5.5197, val loss 5.4838


Training:  10%|█         | 50/500 [01:56<08:23,  1.12s/it] 

step 50: train loss 4.5992, val loss 4.4681


Training:  20%|██        | 100/500 [03:12<07:31,  1.13s/it]

...

Training:  80%|████████  | 400/500 [10:51<01:54,  1.15s/it]

step 400: train loss 2.4926, val loss 2.3982


Training:  90%|█████████ | 450/500 [12:09<00:57,  1.15s/it]

step 450: train loss 2.4580, val loss 2.3999


Training: 100%|██████████| 500/500 [13:07<00:00,  1.58s/it]


In [13]:
lora_model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(lora_model.generate(context, max_new_tokens=500)[0].tolist()))


 bar te
fer aof Ilond t I tok isund frless mme allaowowhy s y tome athithinwalyne b plkese th oshug e cafoaved
, ldlangran y t. heech Ld; avaivhiass anitnd De the.



 Anerour dobsr,is ore wave cay g And s had d bondg d Fve Ty w; awen hangaf, p I d,

rsindove lishefe w
ave ce:


de eerre aw.

se h s 
lvoy
I
d, h bbe t telooonje mp aro

thoush thy hive,

in lddoe he at An. aves vesthe al t
chelith tlbald the the he uthakeisande s ldo thebehea arf in, be An t; oGn, I: eron henchaech mhe isoof they


In [14]:
# base model generation
from train import GPTLanguageModel
base_model = GPTLanguageModel()
base_model.load_state_dict(checkpoint['model_state_dict'])
base_model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(base_model.generate(context, max_new_tokens=500)[0].tolist()))


Ballaber, winters and dying lourning,
In hour sudden as gentle as peaceful stain,
Whose masters peril incense: such debts yet if any of here,
I look forth and to have pardon swift love;
Which on myself in I speak, grave, and gave me:
But to do my father's elders died, and lians,
Like hardness to the like preter action of sweet?
Why, dost thou expercising wase wint to questiathe?
This day int that of thy wold; there liest thy death,
I know the corrupt out.

HENRY BOLINGBROKE:
Exto they shall answ
